# MuonClip quotient-spectrum RG test

This notebook tests whether the apparently random radial bulk of a MuonClip-trained weight matrix can mask a structured RG flow.

For a saved initialization matrix

$$
W_0 = U_0 \Sigma_0 V_0^\top,
$$

define the initial Gram metric

$$
G_0 = W_0^\top W_0
$$

for a square or tall matrix. The right-whitened matrix is

$$
Y_t = W_t G_0^{-1/2}.
$$

At initialization,

$$
Y_0 = W_0 (W_0^\top W_0)^{-1/2} = U_0 V_0^\top,
$$

so every nonzero singular value of $Y_0$ is exactly one. Thus

$$
Y_0^\top Y_0 = I.
$$

The quotient spectrum

$$
\Lambda_t = \operatorname{spec}\left[G_0^{-1/2} W_t^\top W_t G_0^{-1/2}\right]
$$

measures growth or contraction relative to the **initial random metric**, rather than relative to Euclidean coordinates.

For a wide matrix the notebook uses the left Gram metric

$$
G_0 = W_0 W_0^\top
$$

and whitens on the left. In either case the nonzero quotient eigenvalues are the squared singular values of $Y_t$.

The central test is:

$$
\rho(W_t^\top W_t) \approx \rho_{\mathrm{random}}
$$

while

$$
\rho(Y_t^\top Y_t)
$$

may broaden and develop non-random structure.

We compare the exact saved **initial**, **best**, and **final** checkpoints from the same run, and we also include an independent matched Gaussian matrix as a control for artifacts introduced by whitening.


## Important interpretation

Whitening can itself amplify directions in which $W_0$ has unusually small singular values. Therefore a heavy tail in the quotient spectrum is **not** automatically evidence of learned RG structure.

We test this in two ways.

First, we compare the trained quotient spectrum to an independent Gaussian control transformed by the **same** initial metric.

Second, we repeat the quotient analysis with a ridge-regularized metric,

$$
G_{0,\epsilon}^{-1/2} = \left(G_0+\epsilon I\right)^{-1/2},
$$

where

$$
\epsilon = \epsilon_{\mathrm{rel}}\,\frac{\operatorname{Tr}G_0}{d}.
$$

A learned tail is much more convincing if its alpha and shape remain stable over a reasonable range of $\epsilon_{\mathrm{rel}}$, while the randomized control does not show the same behavior.


In [ ]:
import os

RUN_DIR = os.environ.get(
    "RUN_DIR",
    "/tmp/rg-nanogpt-muonclip-3ep-seed4242-20260814_090245/results/muon_clip/seed_4242",
)
TARGET_SEED = int(os.environ.get("TARGET_SEED", "4242"))
TARGET_OPTIMIZER = os.environ.get("TARGET_OPTIMIZER", "muon_clip")
RG_MATRIX_NAME = os.environ.get("RG_MATRIX_NAME", "L00_W_Q")

# Exact whitening by default.  The sensitivity study below tests nonzero ridge values.
WHITEN_EPS_REL = float(os.environ.get("WHITEN_EPS_REL", "0.0"))

# The notebook always displays plots inline.
SHOW_PLOTS = True

print("RUN_DIR        =", RUN_DIR)
print("TARGET_SEED    =", TARGET_SEED)
print("TARGET_OPTIMIZER =", TARGET_OPTIMIZER)
print("RG_MATRIX_NAME =", RG_MATRIX_NAME)
print("WHITEN_EPS_REL =", WHITEN_EPS_REL)


In [ ]:
from pathlib import Path
import sys
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from IPython.display import display, Image

# Make local src importable when launched from baseline/nanogpt_one_head.
ROOT = Path.cwd().resolve()
if (ROOT / "src" / "rg_nanogpt_one_head").is_dir():
    EXPERIMENT_ROOT = ROOT
else:
    candidates = [p for p in [ROOT, *ROOT.parents] if (p / "baseline" / "nanogpt_one_head" / "src" / "rg_nanogpt_one_head").is_dir()]
    if not candidates:
        raise FileNotFoundError("Launch from the rg_optimizers repository or baseline/nanogpt_one_head.")
    EXPERIMENT_ROOT = candidates[0] / "baseline" / "nanogpt_one_head"

sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))

from rg_nanogpt_one_head.angular_weightwatcher_core import (
    AnalysisConfig,
    resolve_run,
    _load_payload,
    _model_config,
    _build_model,
)
from rg_nanogpt_one_head.model import transformer_matrix_items

RUN_PATH = Path(RUN_DIR).expanduser().resolve()
CONFIG = AnalysisConfig(
    seed=TARGET_SEED,
    optimizer=TARGET_OPTIMIZER,
    run_dir=str(RUN_PATH),
    show_plots=True,
)

resolved = resolve_run(CONFIG)
BEST_PATH = RUN_PATH / "checkpoint_best.pt"
if not BEST_PATH.is_file():
    raise FileNotFoundError(BEST_PATH)

CHECKPOINTS = {
    "initial": resolved.initial_path,
    "best": BEST_PATH,
    "final": resolved.final_path,
}

payloads = {name: _load_payload(path) for name, path in CHECKPOINTS.items()}
model_cfg = _model_config(payloads["initial"], payloads["final"], RUN_PATH)

def matrix_from_payload(payload, matrix_name):
    model = _build_model(payload, model_cfg)
    items = {
        name: weight.detach().float().cpu().numpy().astype(np.float64, copy=True)
        for name, _, _, weight in transformer_matrix_items(model)
    }
    del model
    gc.collect()
    if matrix_name not in items:
        raise KeyError(f"{matrix_name!r} not found. Available: {sorted(items)}")
    return items[matrix_name]

W = {state: matrix_from_payload(payload, RG_MATRIX_NAME) for state, payload in payloads.items()}

print("Checkpoints:")
for state, path in CHECKPOINTS.items():
    print(f"  {state:7s} step={int(payloads[state].get('step', -1)):7d}  {path}")

print("\nMatrix shape:", W["initial"].shape)
assert all(W[state].shape == W["initial"].shape for state in W)


## Construct the quotient spectrum

Let the initial matrix have shape $m\times n$.

If $m\ge n$, use right whitening:

$$
Y_t = W_t (W_0^\top W_0+\epsilon I)^{-1/2}.
$$

If $m<n$, use left whitening:

$$
Y_t = (W_0W_0^\top+\epsilon I)^{-1/2}W_t.
$$

This always whitens the smaller Gram matrix and avoids introducing unnecessary zero modes.

For $\epsilon=0$, the initialization quotient spectrum should collapse numerically to

$$
\lambda_i(Y_0^\top Y_0)=1.
$$


In [ ]:
def whitening_operator(W0, eps_rel=0.0):
    m, n = W0.shape
    if m >= n:
        G0 = W0.T @ W0
        side = "right"
    else:
        G0 = W0 @ W0.T
        side = "left"

    evals, evecs = np.linalg.eigh(G0)
    evals = np.maximum(evals, 0.0)
    scale = float(np.mean(evals))
    eps = float(eps_rel) * scale
    denom = np.sqrt(evals + eps)

    if np.any(denom <= 0):
        raise np.linalg.LinAlgError(
            "Initial Gram matrix is singular. Use WHITEN_EPS_REL > 0."
        )

    invsqrt = (evecs * (1.0 / denom)) @ evecs.T
    return invsqrt, side, evals, eps

def whiten_matrix(Wt, invsqrt, side):
    return Wt @ invsqrt if side == "right" else invsqrt @ Wt

G0_inv_sqrt, WHITEN_SIDE, G0_EVALS, EPS_ABS = whitening_operator(
    W["initial"], WHITEN_EPS_REL
)

Y = {
    state: whiten_matrix(W[state], G0_inv_sqrt, WHITEN_SIDE)
    for state in W
}

def esd(M):
    s = np.linalg.svd(M, compute_uv=False)
    return np.sort(s * s)

RAW_ESD = {state: esd(W[state]) for state in W}
Q_ESD = {state: esd(Y[state]) for state in Y}

print("whitening side =", WHITEN_SIDE)
print("absolute ridge =", EPS_ABS)
print("initial Gram condition number =", G0_EVALS.max() / G0_EVALS.min())
print("initial quotient eigenvalue min/max =", Q_ESD["initial"].min(), Q_ESD["initial"].max())
print("max |lambda(Y0)-1| =", np.max(np.abs(Q_ESD["initial"] - 1.0)))


## Matched Gaussian control

The control matrix has the same shape, mean, and standard deviation as $W_0$, but is statistically independent:

$$
W_{\mathrm{G}}\sim\mathcal N(\mu_0,\sigma_0^2).
$$

It is transformed by the **same** whitening operator derived from $W_0$:

$$
Y_{\mathrm{G}}=W_{\mathrm{G}}G_0^{-1/2}.
$$

This control is important because an inverse random metric can itself generate broad generalized-eigenvalue statistics. If the trained quotient tail is indistinguishable from this control, the apparent heavy tail is likely a whitening artifact rather than a learned RG signal.


In [ ]:
rng = np.random.default_rng(91337)
W_gaussian = rng.normal(
    loc=float(W["initial"].mean()),
    scale=float(W["initial"].std()),
    size=W["initial"].shape,
)
Y_gaussian = whiten_matrix(W_gaussian, G0_inv_sqrt, WHITEN_SIDE)

RAW_ESD["gaussian"] = esd(W_gaussian)
Q_ESD["gaussian"] = esd(Y_gaussian)

print("Gaussian raw ESD range     :", RAW_ESD["gaussian"].min(), RAW_ESD["gaussian"].max())
print("Gaussian quotient ESD range:", Q_ESD["gaussian"].min(), Q_ESD["gaussian"].max())


## WeightWatcher alpha and RAND distance

For each raw and quotient matrix we wrap the matrix in a one-layer PyTorch model and run WeightWatcher directly.

The call is deliberately the familiar WeightWatcher call:

```python
watcher.analyze(
    plot=True,
    savefig=savedir,
    min_evals=20,
    randomize=True,
    ERG=False,
)
```

The generated WeightWatcher plots are saved **and displayed inline below**.

For the exact quotient initialization, all singular values are one. A power-law alpha is therefore not mathematically meaningful; WeightWatcher may return `NaN`, a failed fit, or a numerically unstable value. That is expected and is explicitly reported rather than interpreted as an RG exponent.


In [ ]:
import weightwatcher as ww

class OneMatrixModel(nn.Module):
    def __init__(self, name, matrix):
        super().__init__()
        matrix = np.asarray(matrix, dtype=np.float32)
        layer = nn.Linear(matrix.shape[1], matrix.shape[0], bias=False)
        with torch.no_grad():
            layer.weight.copy_(torch.from_numpy(matrix))
        self.add_module(name, layer)

PLOT_ROOT = RUN_PATH / "diagnostics" / "quotient_spectrum_rg" / RG_MATRIX_NAME
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

def run_ww(label, matrix):
    savedir = PLOT_ROOT / label
    savedir.mkdir(parents=True, exist_ok=True)
    watcher = ww.WeightWatcher(model=OneMatrixModel(RG_MATRIX_NAME, matrix))
    try:
        details = watcher.analyze(
            plot=True,
            savefig=str(savedir),
            min_evals=20,
            randomize=True,
            ERG=False,
        )
        row = details.iloc[0].to_dict() if len(details) else {}
        status = "ok"
    except Exception as exc:
        details = pd.DataFrame()
        row = {}
        status = f"{type(exc).__name__}: {exc}"
    return {
        "label": label,
        "status": status,
        "alpha": row.get("alpha", np.nan),
        "D": row.get("D", np.nan),
        "rand_distance": row.get("rand_distance", np.nan),
        "details": details,
        "savedir": savedir,
    }

ww_results = []
for state in ("initial", "best", "final"):
    ww_results.append(run_ww(f"raw_{state}", W[state]))
for state in ("initial", "best", "final"):
    ww_results.append(run_ww(f"quotient_{state}", Y[state]))
ww_results.append(run_ww("raw_gaussian", W_gaussian))
ww_results.append(run_ww("quotient_gaussian", Y_gaussian))

WW_TABLE = pd.DataFrame([
    {k: v for k, v in item.items() if k not in {"details", "savedir"}}
    for item in ww_results
])
display(WW_TABLE)

WW_TABLE.to_csv(PLOT_ROOT / "weightwatcher_alpha_summary.csv", index=False)
print("saved:", PLOT_ROOT / "weightwatcher_alpha_summary.csv")


## Original WeightWatcher ESD plots

These are the actual files emitted by `WeightWatcher.analyze(plot=True, savefig=...)`.

The notebook **always displays them inline**. The most important comparisons are:

- raw initial vs raw best/final;
- quotient initial vs quotient best/final;
- quotient trained vs quotient matched-Gaussian control.

The exact quotient initialization should look like a narrow spike at $\lambda=1$.


In [ ]:
def find_esd_images(savedir):
    files = sorted(savedir.rglob("*.png"))
    esd = [p for p in files if "esd" in p.name.lower()]
    return esd if esd else files

for item in ww_results:
    images = find_esd_images(item["savedir"])
    print("\n", "=" * 80)
    print(item["label"], "status:", item["status"])
    print("directory:", item["savedir"])
    if not images:
        print("No PNG files were generated.")
        continue
    for path in images:
        print(path.name)
        display(Image(filename=str(path)))


## Direct log-log ESD comparison on common bins

WeightWatcher plots are preserved above. This plot adds a controlled comparison using identical logarithmic bins for every spectrum.

The quotient panel is the key test. If MuonClip learning is hidden by the random radial bulk, the raw curves can remain close while the quotient best/final curves separate strongly from both the initialization spike and the independent-Gaussian quotient control.


In [ ]:
def density_on_log_bins(values_dict, nbins=24):
    positive = np.concatenate([
        np.asarray(v)[np.asarray(v) > 0] for v in values_dict.values()
    ])
    lo, hi = positive.min(), positive.max()
    edges = np.geomspace(lo, hi, nbins + 1)
    centers = np.sqrt(edges[:-1] * edges[1:])
    out = {}
    for name, vals in values_dict.items():
        rho, _ = np.histogram(vals, bins=edges, density=True)
        out[name] = rho
    return centers, out

fig, ax = plt.subplots(figsize=(10, 6))
centers, densities = density_on_log_bins(RAW_ESD)
for state, rho in densities.items():
    mask = rho > 0
    ax.loglog(centers[mask], rho[mask], marker="o", label=state)
ax.set_xlabel(r"$\lambda=\sigma^2$")
ax.set_ylabel(r"$\rho(\lambda)$")
ax.set_title(f"{RG_MATRIX_NAME}: raw ESD")
ax.legend()
ax.grid(True, which="both", alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
centers, densities = density_on_log_bins(Q_ESD)
for state, rho in densities.items():
    mask = rho > 0
    ax.loglog(centers[mask], rho[mask], marker="o", label=state)
ax.axvline(1.0, linestyle=":", label="exact quotient initialization: lambda=1")
ax.set_xlabel(r"generalized eigenvalue $\lambda$")
ax.set_ylabel(r"$\rho(\lambda)$")
ax.set_title(f"{RG_MATRIX_NAME}: initial-metric quotient ESD")
ax.legend()
ax.grid(True, which="both", alpha=0.25)
plt.show()


## Alpha summary

The quantity to watch is **not** the quotient-initial alpha: the exact quotient initialization is a delta function at one and has no meaningful heavy-tail exponent.

The informative comparisons are:

$$
\alpha_{\mathrm{raw,best/final}}
$$

versus

$$
\alpha_{\mathrm{quotient,best/final}},
$$

and then

$$
\alpha_{\mathrm{quotient,best/final}}
$$

versus

$$
\alpha_{\mathrm{quotient,Gaussian}}.
$$

If whitening merely manufactures a heavy tail, the trained quotient and Gaussian quotient should look similar. If the trained quotient has a stable and distinct tail, that supports the hypothesis that the raw MP-like bulk is masking learned structure.


In [ ]:
summary = WW_TABLE.set_index("label")
display(summary[["alpha", "D", "rand_distance", "status"]])

fig, ax = plt.subplots(figsize=(11, 5))
labels = list(WW_TABLE["label"])
alphas = pd.to_numeric(WW_TABLE["alpha"], errors="coerce").to_numpy(float)
x = np.arange(len(labels))
ax.scatter(x[np.isfinite(alphas)], alphas[np.isfinite(alphas)], s=70)
ax.axhline(2.0, linestyle="--", label=r"$\alpha=2$")
ax.set_xticks(x, labels, rotation=45, ha="right")
ax.set_ylabel("WeightWatcher alpha")
ax.set_title(f"{RG_MATRIX_NAME}: raw and quotient WeightWatcher alpha")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Ridge sensitivity: does inverse-metric whitening create the tail?

We now vary

$$
\epsilon_{\mathrm{rel}}
$$

over several decades and rerun WeightWatcher on the quotient **best**, **final**, and matched Gaussian matrices.

A robust learned quotient exponent should not disappear under a very small ridge perturbation. Conversely, an apparent heavy tail that is generated almost entirely by the smallest singular value of $W_0$ will move strongly as soon as $\epsilon$ is introduced.


In [ ]:
EPS_GRID = np.array([0.0, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2], dtype=float)

sensitivity_rows = []
for eps_rel in EPS_GRID:
    invsqrt, side, _, eps_abs = whitening_operator(W["initial"], eps_rel)
    matrices = {
        "best": whiten_matrix(W["best"], invsqrt, side),
        "final": whiten_matrix(W["final"], invsqrt, side),
        "gaussian": whiten_matrix(W_gaussian, invsqrt, side),
    }
    for state, matrix in matrices.items():
        result = run_ww(f"sensitivity_eps_{eps_rel:g}_{state}", matrix)
        sensitivity_rows.append({
            "eps_rel": eps_rel,
            "eps_abs": eps_abs,
            "state": state,
            "alpha": result["alpha"],
            "D": result["D"],
            "rand_distance": result["rand_distance"],
            "status": result["status"],
        })

SENSITIVITY = pd.DataFrame(sensitivity_rows)
display(SENSITIVITY)
SENSITIVITY.to_csv(PLOT_ROOT / "whitening_ridge_sensitivity.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 6))
for state in ("best", "final", "gaussian"):
    subset = SENSITIVITY[SENSITIVITY.state == state]
    x = subset.eps_rel.to_numpy(float)
    y = pd.to_numeric(subset.alpha, errors="coerce").to_numpy(float)
    # Plot exact whitening at a small display coordinate left of 1e-10.
    xplot = np.where(x == 0.0, 1e-12, x)
    ax.semilogx(xplot, y, marker="o", label=state)
ax.axhline(2.0, linestyle="--", label=r"$\alpha=2$")
ax.set_xlabel(r"ridge $\epsilon_{\rm rel}$ (0 shown at $10^{-12}$)")
ax.set_ylabel("WeightWatcher alpha")
ax.set_title(f"{RG_MATRIX_NAME}: quotient alpha stability to whitening ridge")
ax.legend()
ax.grid(True, which="both", alpha=0.25)
plt.show()


## Decision rule

The hypothesis that MuonClip's random-looking radial bulk is **masking** an RG flow becomes substantially more plausible if all of the following occur:

1. The raw best/final ESD remains close to the raw initialization or randomized control.
2. The quotient best/final ESD broadens strongly away from the initialization spike at one.
3. The quotient trained spectrum is visibly and statistically different from the quotient independent-Gaussian control.
4. The trained quotient alpha is stable under small ridge values.
5. The Gaussian-control alpha behaves differently under the same ridge sweep.

If the trained and Gaussian quotient spectra are similar, or if the trained alpha collapses as soon as a tiny ridge is introduced, the heavy tail is more likely to be an artifact of dividing by small random singular values in $W_0$.

This notebook deliberately treats the quotient spectrum as a **generalized-eigenvalue experiment** rather than assuming in advance that it must reproduce the ordinary WeightWatcher RG universality class.
